In [15]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers datasets torch tensorboard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

ModuleNotFoundError: No module named 'datasets'

### 1. Data preparation

In [3]:
dataset = load_dataset("dair-ai/emotion")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [4]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [5]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=True, max_length=128)

In [6]:
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [7]:
tokenized_datasets.set_format(type="torch", columns=['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])


### 2. Modeling

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",               # Where to save model and logs
    do_eval=True,                         # Evaluate after each epoch
    save_strategy="epoch",                # Save model after each epoch
    learning_rate=2e-5,                   # Learning rate
    per_device_train_batch_size=16,       # Batch size per device during training
    per_device_eval_batch_size=16,        # Batch size for evaluation
    num_train_epochs=10,                  # Number of training epochs
    weight_decay=0.01,                    # Weight decay
    logging_dir="./logs",                 # Log directory
    logging_steps=50,                     # Log every 50 steps
    report_to="tensorboard",              # Enable TensorBoard logging
)

In [25]:
def compute_metrics(p):
    preds, labels = p
    # Convert predictions to tensors if they are numpy arrays
    preds = torch.tensor(preds) if isinstance(preds, np.ndarray) else preds
    labels = torch.tensor(labels) if isinstance(labels, np.ndarray) else labels

    # Get the predicted class
    preds = torch.argmax(preds, axis=1)

    # Calculate accuracy and F1 score
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")

    return {"accuracy": acc, "f1": f1}

In [26]:
trainer = Trainer(
    model=model,                          # The model to be trained
    args=training_args,                   # The training arguments
    train_dataset=tokenized_datasets["train"],  # The training dataset
    eval_dataset=tokenized_datasets["validation"],  # The validation dataset
    tokenizer=tokenizer,                  # Tokenizer to handle input text
    compute_metrics=compute_metrics,      # Evaluation metrics
)

<ipython-input-26-8e8cb7733f9e>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [27]:
trainer.train()

Step,Training Loss
50,0.175100
100,0.195500
150,0.162100
200,0.141500
250,0.146200
300,0.133000
350,0.115300
400,0.098200
450,0.112400
500,0.136800


TrainOutput(global_step=10000, training_loss=0.05182241513803601, metrics={'train_runtime': 2548.242, 'train_samples_per_second': 62.788, 'train_steps_per_second': 3.924, 'total_flos': 6851123482299840.0, 'train_loss': 0.05182241513803601, 'epoch': 10.0})

### 3. Model evaluation

In [32]:
trainer.evaluate(tokenized_datasets["test"])


Test Results: {'eval_loss': 0.4785479009151459, 'eval_accuracy': 0.9195, 'eval_f1': 0.919868545141683, 'eval_runtime': 7.9222, 'eval_samples_per_second': 252.456, 'eval_steps_per_second': 15.779, 'epoch': 10.0}


In [30]:

model.save_pretrained("/content/emotion_model")
tokenizer.save_pretrained("/content/emotion_model")

('/content/emotion_model/tokenizer_config.json',
 '/content/emotion_model/special_tokens_map.json',
 '/content/emotion_model/vocab.txt',
 '/content/emotion_model/added_tokens.json',
 '/content/emotion_model/tokenizer.json')

In [29]:
model.save_pretrained("/content/drive/MyDrive/emotion_model")
tokenizer.save_pretrained("/content/drive/MyDrive/emotion_model")

('/content/drive/MyDrive/emotion_model/tokenizer_config.json',
 '/content/drive/MyDrive/emotion_model/special_tokens_map.json',
 '/content/drive/MyDrive/emotion_model/vocab.txt',
 '/content/drive/MyDrive/emotion_model/added_tokens.json',
 '/content/drive/MyDrive/emotion_model/tokenizer.json')

### 4. Inference test

In [33]:
from transformers import pipeline
emotion_classifier = pipeline("text-classification", model="/content/emotion_model", tokenizer=tokenizer)

Device set to use cuda:0


In [86]:
label_map = {
    'LABEL_0': 'Fear',
    'LABEL_1': 'Surprise',
    'LABEL_2': 'Love',
    'LABEL_3': 'Anger',
    'LABEL_4': 'Sadness',
    'LABEL_5': 'Joy'}

#  sadness (0), joy (1), love (2), anger (3), fear (4), surprise (5).

In [87]:
texts = ["I hate you", "I fear if he brokes us", "Amazing yoooho", "I love you"]

In [88]:
predictions = emotion_classifier(texts)

In [89]:
labeled_predictions = [
    {"text": text, "predicted_mood": label_map[pred["label"]], "confidence_score": pred["score"]}
    for text, pred in zip(texts, predictions)
]

In [90]:
labeled_predictions

[{'text': 'I hate you',
  'predicted_mood': 'Anger',
  'confidence_score': 0.9998915195465088},
 {'text': 'I fear if he brokes us',
  'predicted_mood': 'Fear',
  'confidence_score': 0.9999686479568481},
 {'text': 'Amazing yoooho',
  'predicted_mood': 'Joy',
  'confidence_score': 0.9968936443328857},
 {'text': 'I love you',
  'predicted_mood': 'Love',
  'confidence_score': 0.9931267499923706}]